In [1]:
import pandas as pd
import numpy as np
import pickle
import random
import os

In [2]:
print("📁 Current Path:", os.getcwd())

# files check karo
print("Files in root:", os.listdir())

if os.path.exists("notebook"):
    print("Files in notebook folder:", os.listdir("notebook"))

📁 Current Path: /workspaces/project.final_allocation-3/notebook
Files in root: ['final_model.pkl', 'ambulance_updated_dataset (3).csv', 'project.ipynb']


In [3]:
model = None

if os.path.exists("final_model.pkl"):
    model = pickle.load(open("final_model.pkl", "rb"))
    print("✅ Loaded from root")

elif os.path.exists("notebook/final_model.pkl"):
    model = pickle.load(open("notebook/final_model.pkl", "rb"))
    print("✅ Loaded from notebook folder")

else:
    raise FileNotFoundError("❌ final_model.pkl not found")

✅ Loaded from root


In [4]:
def compute_risk(traffic, past_demand, rain, temperature, peak_hour):
    return round(
        (traffic * 30) +
        (past_demand * 2) +
        (rain * 1.5) +
        (temperature * 0.5) +
        (peak_hour * 20),
        2
    )

In [5]:
def generate_zones(lat, lon):
    return [
        {"name": "Zone A", "lat": lat + 0.02, "lon": lon + 0.02},
        {"name": "Zone B", "lat": lat - 0.02, "lon": lon + 0.01},
        {"name": "Zone C", "lat": lat + 0.01, "lon": lon - 0.02},
        {"name": "Zone D", "lat": lat - 0.01, "lon": lon - 0.01},
    ]

In [6]:
def simulate_zone_risk(base_inputs, zones):

    results = []

    for z in zones:
        traffic = min(1, max(0, base_inputs['traffic_level'] + random.uniform(-0.1, 0.1)))
        rain = max(0, base_inputs['rain'] + random.uniform(-2, 2))
        temp = base_inputs['temperature'] + random.uniform(-2, 2)

        risk = compute_risk(
            traffic,
            base_inputs['past_1hr_demand'],
            rain,
            temp,
            base_inputs['peak_hour']
        )

        results.append({
            "zone": z["name"],
            "latitude": z["lat"],
            "longitude": z["lon"],
            "risk": risk
        })

    return sorted(results, key=lambda x: x["risk"], reverse=True)

In [7]:
def smart_system(input_data):

    df_input = pd.DataFrame([input_data])

    prediction = model.predict(df_input)[0]

    hour = input_data['hour']
    peak_hour = 1 if (8 <= hour <= 11 or 17 <= hour <= 21) else 0

    risk_score = compute_risk(
        input_data['traffic_level'],
        input_data['past_1hr_demand'],
        input_data['rain'],
        input_data['temperature'],
        peak_hour
    )

    # ambulance logic
    if prediction == 1 and input_data['traffic_level'] > 0.7:
        ambulances = 3
        level = "HIGH"
    elif prediction == 1:
        ambulances = 2
        level = "MEDIUM"
    else:
        ambulances = 1
        level = "LOW"

    response_time = 10 + (input_data['traffic_level'] * 10)

    zones = generate_zones(input_data['latitude'], input_data['longitude'])

    base_inputs = {
        "traffic_level": input_data['traffic_level'],
        "past_1hr_demand": input_data['past_1hr_demand'],
        "rain": input_data['rain'],
        "temperature": input_data['temperature'],
        "peak_hour": peak_hour
    }

    zone_risks = simulate_zone_risk(base_inputs, zones)

    return {
        "prediction": int(prediction),
        "priority": level,
        "ambulances": ambulances,
        "response_time": round(response_time, 2),
        "risk_score": risk_score,
        "best_zone": zone_risks[0],
        "backup_zone": zone_risks[1]
    }

In [8]:
sample = {
    "latitude": 19.07,
    "longitude": 72.87,
    "emergency_type": "EMS",
    "hour": 18,
    "day": "Friday",
    "traffic_level": 0.8,
    "temperature": 32,
    "rain": 5,
    "past_1hr_demand": 10,
    "peak_hour": 1
}

result = smart_system(sample)
print(result)

{'prediction': 1, 'priority': 'HIGH', 'ambulances': 3, 'response_time': 18.0, 'risk_score': 87.5, 'best_zone': {'zone': 'Zone C', 'latitude': 19.080000000000002, 'longitude': 72.85000000000001, 'risk': 87.65}, 'backup_zone': {'zone': 'Zone A', 'latitude': 19.09, 'longitude': 72.89, 'risk': 85.91}}


In [9]:
def detect_surge(past_demand, traffic, peak_hour):

    surge_score = (past_demand * 1.5) + (traffic * 50) + (peak_hour * 30)

    if surge_score > 120:
        return "HIGH SURGE ⚠️"
    elif surge_score > 80:
        return "MEDIUM SURGE"
    else:
        return "NO SURGE"